# 03. Train XGBoost Model
In this notebook, we load the engineered features and train the XGBoost Regressor.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data
df = pd.read_parquet('../dataset/processed/engineered_features.parquet')
print(f"Loaded {len(df)} rows for training.")

# Define features and target
features_cols = ['lag_demand_1h', 'lag_demand_2h', 'lag_demand_24h', 'lag_demand_168h', 
                 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
target_col = 'total_demand'

# Simple time-based split
train_df = df.iloc[:int(len(df)*0.8)]
test_df = df.iloc[int(len(df)*0.8):]

X_train, y_train = train_df[features_cols], train_df[target_col]
X_test, y_test = test_df[features_cols], test_df[target_col]

# Train XGBoost
model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=7, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")

# Save model
os.makedirs('../saved_models/xgboost', exist_ok=True)
model.save_model('../saved_models/xgboost/xgboost_demand_model.json')
print("Model saved to ../saved_models/xgboost/xgboost_demand_model.json")